# Suite SKL — Agent guides

`AGENTS.md`, `llms.txt` and the skill in `skills/mbse-expressions/` teach AI agents what this package is for and how
to use it. Stale guidance misleads more than missing guidance, so these cases check that the packaged copy of the
skill is current, that its complete programs run, and that every link in the guides resolves.

In [ ]:
import { spawnSync } from "node:child_process";
import { existsSync, mkdirSync, readdirSync, readFileSync, statSync, writeFileSync } from "node:fs";
import { join, relative, resolve } from "node:path";

import { assert } from "./support.js";

const ROOT = resolve("../..");
const SKILL = join(ROOT, "skills", "mbse-expressions");
const PACKAGED = resolve("../skill");
const REPO = "https://github.com/pitaman71/mbse-expressions/blob/main/";

function walk(folder: string): string[] {
  return readdirSync(folder).flatMap((name) => {
    const path = join(folder, name);
    return statSync(path).isDirectory() ? walk(path) : [path];
  });
}

/** The files under `folder`, by relative path, in code-point order. */
function files(folder: string): [string, string][] {
  return walk(folder).map((path): [string, string] => [relative(folder, path).split("\\").join("/"), readFileSync(path, "latin1")])
    .sort(([a], [b]) => (a < b ? -1 : a > b ? 1 : 0));
}

const GUIDES = [join(ROOT, "AGENTS.md"), join(ROOT, "llms.txt"),
  ...files(SKILL).filter(([name]) => name.endsWith(".md")).map(([name]) => join(SKILL, name))];

/** The complete programs in the skill: fenced blocks whose info string is exactly `language`. */
function blocks(language: string): string[] {
  return GUIDES.slice(2).flatMap((guide) => [...readFileSync(guide, "utf8").matchAll(new RegExp("^```" + language + "\\n(.*?)^```$", "gms"))]
    .map((match) => match[1] as string));
}

const LINKS = 38;

## SKL-01 · The packaged copy of the skill is current

In [ ]:
assert(files(SKILL).some(([name]) => name === "SKILL.md"));
assert(JSON.stringify(files(PACKAGED)) === JSON.stringify(files(SKILL)), "the packaged skill is stale: run skills/sync.sh");

## SKL-02 · Every complete program in the skill runs

In [ ]:
{
  const programs = blocks("typescript");
  assert(programs.length === 1, String(programs.length));
  mkdirSync(".build", { recursive: true });
  const failures: Record<number, string> = {};
  programs.forEach((code, i) => {
    const file = `.build/skill-${i}.ts`;
    writeFileSync(file, code);
    const check = spawnSync("npx", ["tsc", "--noEmit", "--strict", "--skipLibCheck", "--module", "nodenext",
      "--moduleResolution", "nodenext", "--target", "es2022", file], { encoding: "utf8", timeout: 300_000 });
    const run = check.status === 0 ? spawnSync("npx", ["tsx", file], { encoding: "utf8", timeout: 300_000 }) : check;
    if (run.status !== 0) failures[i] = `${run.stdout || ""}${run.stderr || ""}`.slice(-2000);
  });
  assert(Object.keys(failures).length === 0, JSON.stringify(failures));
}

## SKL-03 · Every link in the agent guides resolves, anchors included

In [ ]:
{
  /** GitHub's anchor for a heading. */
  const slug = (heading: string) => heading.trim().toLowerCase().replace(/[^\w\- ]/g, "").split(" ").join("-");
  const problems: string[] = [];
  let checked = 0;
  for (const guide of GUIDES) {
    for (const [, target] of readFileSync(guide, "utf8").matchAll(/\]\(([^)\s]+)\)/g) as IterableIterator<[string, string]>) {
      let path: string, base: string;
      if (target.startsWith(REPO)) [path, base] = [target.slice(REPO.length), ROOT];
      else if (target.includes("://")) continue;
      else [path, base] = [target, resolve(guide, "..")];
      const [name, anchor] = path.split("#") as [string, string | undefined];
      const resolved = join(base, name);
      checked += 1;
      const label = `${relative(resolve(guide, ".."), guide)}: ${target}`;
      if (!existsSync(resolved)) problems.push(label);
      else if (anchor && ![...readFileSync(resolved, "utf8").matchAll(/^#+ (.*)$/gm)].some((m) => slug(m[1] as string) === anchor)) {
        problems.push(label);
      }
    }
  }
  assert(problems.length === 0, problems.join("; "));
  assert(checked === LINKS, String(checked));
}